# Invasion detection

Using the co-culture data with phalloidin staining that show hyphal invasion in epithelial cells, this notebook tries to distinguish invaded cells from non-invaded.

## File Import

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_01_CY5, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_02_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_DIC",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_FITC, BF",
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

config_DIC = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit",
        "correct_DIC_shift": True
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[3], file_pattern[0], verbose=True)
files2 = find_files_by_pattern(search_path[4],file_pattern[2],verbose=True)
# containers = [ImageContainer(f, config) for f in files]

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/label_invasion")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

In [ ]:
cy5 = ImageContainer(files[0],config)
dic = ImageContainer(files2[0],config_DIC)

## Plot overlay image

Overlay all layers

In [ ]:
import gc
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)

containers[0].display(ax=axes[0,0])
containers[1].display(ax=axes[0,1])
containers[2].display(ax=axes[1,0])
total = ImageContainer(files,config)
total.display(ax=axes[1,1])
axes[0,0].set_title('Phalloidin')
axes[0,1].set_title('DAPI')
axes[1,0].set_title('DIC')
axes[1,1].set_title('Combined')

output_filename = current_output_dir / f"{Path(total.name).stem}_display.png"
plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

## Run edge detection

In [ ]:
from image_processing_tools.util.phalloidin_processing import process_actin_fibers
import gc
import matplotlib.pyplot as plt

arrow_params = {
    'flip_opposing_vectors': False,
    'use_log_magnitude': True,
    'clip_magnitude': True,
}

cy5_edges = process_actin_fibers(cy5.merge(), edge_method = "sobel", **arrow_params)

output_filename = current_output_dir / f"{Path(cy5.name).stem}_edge_detection.png"
plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor

pipeline = ImageJProcessor(dic.merge())
pipeline.reset()
pipeline.enhance_contrast_clahe(block_size=127, slope=3, bins=256)
pipeline.fft_bandpass_filter(large_structures_down_to=60, small_structures_up_to=3, suppress_stripes='Horizontal', tolerance=0.05)
pipeline.subtract_background_imagej_exact(radius=50, light_bg=False)
vesselness, scale_map = pipeline.frangi_imagej_ops(scales=[8,10],c=1,apply_gaussian=False)

# output_filename = current_output_dir / f"{Path(dic.name).stem}_edge_detection.png"
# plt.savefig(output_filename, dpi=300, bbox_inches='tight')
# plt.close()
# gc.collect()
# print(f"Saved: {output_filename}")

## Multiplication

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from image_processing_tools.util.phalloidin_processing import get_otsu_edge_map
from skimage import img_as_float
from scipy.ndimage import binary_dilation

def get_mag(edges):
    gx, gy = edges
    mag = np.sqrt(gx**2 + gy**2)
    return mag

cy5_mag = get_mag(cy5_edges)
cy5_mag = img_as_float(cy5_mag)
cy5_mag = get_otsu_edge_map(cy5_mag)
cy5_mag = binary_dilation(cy5_mag, iterations=2)
vessel_mag = get_otsu_edge_map(vesselness)
vessel_mag = binary_dilation(vessel_mag, iterations=2)

multi = cy5_mag*vessel_mag
# multi = get_otsu_edge_map(multi)

fig,axes = plt.subplots(1,3,figsize=(4,12))

axes[0].imshow(cy5_mag)
axes[0].set_title('Phalloidin Edges')
axes[1].imshow(vessel_mag)
axes[1].set_title('DIC Edges')
axes[2].imshow(multi)
axes[2].set_title('Multiplication')
for ax in axes:
    ax.axis('off')
plt.show()
output_filename = current_output_dir / f"{Path(cy5.name).stem}_edge_detection_multiply.png"
plt.savefig(output_filename, dpi=600, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

## Gradient vectors colored by magnitude vs. direction

In [ ]:
import matplotlib.pyplot as plt
from image_processing_tools.util.phalloidin_processing import plot_gradient_field_on_empty_axis

# Assuming gx and gy are already defined in your environment
# and plot_gradient_field_on_empty_axis is imported/available

fig, axes = plt.subplots(1,2,figsize=(12, 6))

# Parameters to make arrows smaller and denser
arrow_params = {
    'step': 10,              # Smaller step = more arrows (default was 20 or 50)
    'arrow_scale': 0.8,      # Scale of arrow length relative to the step size
    'arrow_width': 0.001,    # Thinner arrows to avoid cluttering
    'alpha': 0.8,
    'rotate_90': True,       # Standard rotation for this pipeline
    'color_by_magnitude': False,
    'use_log_magnitude': True,
    'colormap': 'viridis',
    'flip_opposing_vectors': False,
    'clip_magnitude': False,
}

plot_gradient_field_on_empty_axis(axes[0], gx, gy, **arrow_params)
axes[0].set_title('Color by Direction')

arrow_params['color_by_magnitude'] = True
plot_gradient_field_on_empty_axis(axes[1], gx, gy, **arrow_params)
axes[1].set_title('Color by Magnitude')

plt.tight_layout()
plt.show()

output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_gradient_only.png"

plt.savefig(output_filename, dpi=450, bbox_inches='tight')
plt.close(fig)
gc.collect()
print(f"Saved: {output_filename}")

## Magnitude Histogram

In [ ]:
from image_processing_tools.util.phalloidin_processing import plot_gradient_magnitude_histogram

plot_gradient_magnitude_histogram(gx, gy, use_log_magnitude = True, log_frequency = False, clip_magnitude=True)

output_filename = current_output_dir / f"{Path(fitc_container.name).stem}_magnitude_histogram.png"

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

## Visualize gradient vectors in 3D

In [ ]:
from image_processing_tools.util.phalloidin_processing import plot_3d_gradient_magnitude_plotly
from pathlib import Path

downsample_ratio = 4
output_filename = current_output_dir / f"{Path(fitc_container.name).stem}_downsample_{downsample_ratio}x_3D_surface.html"

plot_3d_gradient_magnitude_plotly(gx, gy, 
                                  downsample=downsample_ratio, 
                                  flip_opposing_vectors=False, 
                                  smooth_sigma = 10, 
                                  smooth_angle_sigma = 3,
                                  use_log_magnitude=True,
                                  save_path = output_filename,
                                  show_plot = False,
                                  clip_magnitude=True)

## Fit gradient magnitudes to a GMM classifier

In [ ]:
from image_processing_tools.util.phalloidin_processing import fit_gradient_magnitude_gmm

gmm_plot_args = {
    'n_components':2, 
    'use_log_magnitude': True, 
    'clip_magnitude': True,
    'show_plot': True,
    'prob_colormap': 'uncertainty'
}

fit_gradient_magnitude_gmm(gx, gy, **gmm_plot_args)

output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_{gmm_plot_args.get('prob_colormap')}_gmm_fit.png"

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")